# Candidate Detection with IR-MAD

IR-MAD based optical and SAR change fusion, then SLIC + hysteresis + blob ranking.

In [ ]:
import os
import re
import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, transform_bounds, Resampling
from rasterio.windows import Window, transform as window_transform
from scipy.stats import chi2
from skimage.segmentation import slic
from skimage.filters import apply_hysteresis_threshold
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects
from huggingface_hub import HfApi, hf_hub_download, CommitOperationAdd
from kaggle_secrets import UserSecretsClient

HF_REPO_ID = 'sasudo2/landslides'
HF_REPO_TYPE = 'dataset'
HF_REVISION = 'main'
HF_RAW_ROOT = 'raw_images/raw_incidents'
HF_CAND_ROOT = 'candidates'
HIGH_CONF = 0.999
LOW_CONF = 0.90
MIN_SLOPE_DEG = 20
MAX_ITERS = 20
MAX_CANDIDATES_PER_INCIDENT = 100
MIN_BLOB_AREA_M2 = 2000
MAX_BLOB_AREA_M2 = 2000000
MAX_ELONGATION = 6.0
CHIP_PAD_M = 100   # padding (meters) added around each candidate's bbox for the exported chips
UPLOAD_BATCH_SIZE = 25   # chip files per batched HF commit

# Common analysis grid: everything (before/after/SAR/slope) is reprojected onto this
# single grid so pixels correspond to the same ground location. Target resolution
# matches GEE's native Sentinel-2/SRTM sampling (~10 m) per the pipeline spec.
GEE_SCALE_M = 10
TARGET_CRS = 'EPSG:4326'
TARGET_SUPERPIXEL_M = 30   # desired physical superpixel footprint, drives SLIC segment count
M_PER_DEG_LAT = 111320.0   # meters per degree of latitude, ~constant everywhere

def norm01(x):
    a = np.nanmin(x); b = np.nanmax(x)
    if not np.isfinite(a) or not np.isfinite(b) or b <= a:
        return np.zeros_like(x, dtype=np.float32)
    return ((x-a)/(b-a)).astype(np.float32)

def wmean_cov(X, w):
    w = w / np.sum(w)
    m = np.sum(X * w[:, None], axis=0)
    Xc = X - m
    C = (Xc * w[:, None]).T @ Xc
    return m, C

def cca_axes(Sxx, Syy, Sxy, reg=1e-6):
    p = Sxx.shape[0]
    q = Syy.shape[0]
    Sxx = Sxx + reg * np.eye(p)
    Syy = Syy + reg * np.eye(q)
    invSxx = np.linalg.inv(Sxx)
    invSyy = np.linalg.inv(Syy)
    M = invSxx @ Sxy @ invSyy @ Sxy.T
    vals, vecs = np.linalg.eigh(M)
    idx = np.argsort(vals)[::-1]
    A = vecs[:, idx]
    B = invSyy @ Sxy.T @ A
    for i in range(B.shape[1]):
        d = np.sqrt(B[:, i].T @ Syy @ B[:, i])
        if d > 0:
            B[:, i] /= d
    return A, B

def irmad(X, Y, max_iters=MAX_ITERS, tol=1e-3):
    n, p = X.shape
    w = np.ones(n, dtype=np.float64)
    prev = None
    for _ in range(max_iters):
        mx, Sxx = wmean_cov(X, w)
        my, Syy = wmean_cov(Y, w)
        Xc = X - mx
        Yc = Y - my
        wn = w / np.sum(w)
        Sxy = (Xc * wn[:, None]).T @ Yc
        A, B = cca_axes(Sxx, Syy, Sxy)
        U = Xc @ A[:, :p]
        V = Yc @ B[:, :p]
        M = U - V
        s = np.std(M, axis=0) + 1e-6
        chi = np.sum((M / s[None, :]) ** 2, axis=1)
        conf = 1.0 - chi2.cdf(chi, df=p)
        if prev is not None and np.mean(np.abs(conf - prev)) < tol:
            break
        prev = conf
        w = np.clip(conf, 1e-6, 1.0)
    return chi, conf

def hf_fetch(token, incident_id, fn):
    return hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION, filename=f'{HF_RAW_ROOT}/incident_{incident_id}/{fn}', token=token)

def build_target_grid(ref_path, scale_m=GEE_SCALE_M, dst_crs=TARGET_CRS):
    with rasterio.open(ref_path) as src:
        # Meters-per-degree-longitude shrinks with cos(latitude); using a uniform
        # 111320 m/deg for both axes over-estimates longitude resolution away from the
        # equator (~10-14% at Nepal's ~26-30N). Compute the AOI's mean latitude and use
        # it to scale the longitude (x) resolution separately from latitude (y).
        lon_min, lat_min, lon_max, lat_max = transform_bounds(src.crs, dst_crs, *src.bounds)
        mean_lat_rad = np.deg2rad((lat_min + lat_max) / 2.0)
        m_per_deg_lon = M_PER_DEG_LAT * max(np.cos(mean_lat_rad), 1e-6)
        res_deg_lon = scale_m / m_per_deg_lon
        res_deg_lat = scale_m / M_PER_DEG_LAT
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=(res_deg_lon, res_deg_lat))
    return transform, int(width), int(height), mean_lat_rad

def reproject_bands(path, band_indexes, dst_transform, dst_crs, dst_shape, resampling=Resampling.bilinear):
    h, w = dst_shape
    out = np.zeros((len(band_indexes), h, w), dtype=np.float32)
    with rasterio.open(path) as src:
        for i, band_idx in enumerate(band_indexes):
            reproject(
                source=rasterio.band(src, band_idx),
                destination=out[i],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=resampling,
            )
    return out

def load_optical(before_path, after_path, before_is_s2, dst_transform, dst_crs, dst_shape):
    if before_is_s2:
        # Sentinel-2 band order (1-indexed): B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12,SCL
        before = reproject_bands(before_path, [4, 3, 2, 8], dst_transform, dst_crs, dst_shape, Resampling.bilinear)  # R,G,B,NIR
        scl = reproject_bands(before_path, [13], dst_transform, dst_crs, dst_shape, Resampling.nearest)[0]  # categorical -> nearest only
        valid = np.isin(scl, [2, 4, 5, 6, 7, 11])
    else:
        # Planet analytic_sr_udm2 band order (1-indexed): Blue, Green, Red, NIR
        before = reproject_bands(before_path, [3, 2, 1, 4], dst_transform, dst_crs, dst_shape, Resampling.bilinear)  # R,G,B,NIR
        valid = np.ones(dst_shape, dtype=bool)

    with rasterio.open(after_path) as a:
        after_count = a.count
    after_indexes = [3, 2, 1, 4] if after_count >= 4 else list(range(1, after_count + 1))
    after = reproject_bands(after_path, after_indexes, dst_transform, dst_crs, dst_shape, Resampling.bilinear)

    return before.astype(np.float32), after.astype(np.float32), valid

def load_slope(path, dst_transform, dst_crs, dst_shape):
    return reproject_bands(path, [1], dst_transform, dst_crs, dst_shape, Resampling.bilinear)[0]

def load_sar(pre_path, post_path, dst_transform, dst_crs, dst_shape):
    pre = reproject_bands(pre_path, [1, 2], dst_transform, dst_crs, dst_shape, Resampling.bilinear)
    post = reproject_bands(post_path, [1, 2], dst_transform, dst_crs, dst_shape, Resampling.bilinear)
    return pre, post

def write_chip(path, arr, transform, crs):
    data = arr[None, ...] if arr.ndim == 2 else arr
    meta = {
        'driver': 'GTiff',
        'dtype': 'float32',
        'count': data.shape[0],
        'height': data.shape[1],
        'width': data.shape[2],
        'crs': crs,
        'transform': transform,
    }
    with rasterio.open(path, 'w', **meta) as dst:
        dst.write(data.astype(np.float32))

secrets = UserSecretsClient()
hf_token = secrets.get_secret('huggingface_token')
api = HfApi(token=hf_token)

# Discover which incidents actually have raw imagery uploaded to HF, and which of those
# already have candidate outputs - scan the HF repo directly instead of looping over
# every incident in the source CSV (incident_download.ipynb uploads incrementally and
# out of CSV-row order, so the CSV no longer reflects what's actually ready to process).
try:
    all_repo_files = set(api.list_repo_files(HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION))
except Exception:
    all_repo_files = set()

raw_file_re = re.compile(rf'^{re.escape(HF_RAW_ROOT)}/incident_(\d+)/(.+)$')
raw_files_by_incident = {}
for f in all_repo_files:
    m = raw_file_re.match(f)
    if m:
        raw_files_by_incident.setdefault(int(m.group(1)), set()).add(m.group(2))

MANDATORY_RAW_SUFFIXES = {'after.tif', 'slope.tif', 'sar_pre.tif', 'sar_post.tif'}

def has_mandatory_raw(inc_id, files):
    prefix = f'incident_{inc_id}_'
    suffixes = {fn[len(prefix):] for fn in files if fn.startswith(prefix)}
    if not MANDATORY_RAW_SUFFIXES.issubset(suffixes):
        return False
    return 'planet_before.tif' in suffixes or 'gee_before.tif' in suffixes

incident_ids = sorted(inc_id for inc_id, files in raw_files_by_incident.items() if has_mandatory_raw(inc_id, files))
print(f'Incidents with complete raw imagery on HF: {len(incident_ids)}')

# Skip incidents that already have candidate outputs on HF (idempotent re-runs).
existing_cand_files = {f for f in all_repo_files if f.startswith(f'{HF_CAND_ROOT}/')}

pending_ops = []    # list[CommitOperationAdd] waiting for the next batched HF commit
pending_meta = []   # list[(inc_id, local_dir)] describing what pending_ops holds

def flush_pending():
    global pending_ops, pending_meta
    if not pending_ops:
        return
    n_incidents = len(pending_meta)
    try:
        api.create_commit(
            repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
            operations=pending_ops,
            commit_message=f'Add candidate chips for {n_incidents} incident(s)',
        )
        for inc_id, _ in pending_meta:
            print(f'Uploaded candidate chips for incident_{inc_id} (batch flush of {len(pending_ops)} files / {n_incidents} incidents)')
    except Exception as e:
        for inc_id, _ in pending_meta:
            print(f'Batch upload failed for incident_{inc_id}: {e}')
    finally:
        for _, local_dir in pending_meta:
            shutil.rmtree(local_dir, ignore_errors=True)
        pending_ops = []
        pending_meta = []

records = []
candidate_metadata = []
for inc_id in incident_ids:
    inc_cand_prefix = f'{HF_CAND_ROOT}/incident_{inc_id}/'
    if any(f.startswith(inc_cand_prefix) for f in existing_cand_files):
        print(f'Skip incident_{inc_id}: candidates already exist on HF')
        continue
    try:
        after_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_after.tif')
        slope_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_slope.tif')
        before_is_s2 = False
        try:
            before_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_planet_before.tif')
            before_is_s2 = False
        except Exception:
            before_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_gee_before.tif')
            before_is_s2 = True
        sar_pre = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_sar_pre.tif')
        sar_post = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_sar_post.tif')

        dst_transform, gw, gh, mean_lat_rad = build_target_grid(before_path)
        dst_shape = (gh, gw)
        h, w = dst_shape

        b4, a4, valid = load_optical(before_path, after_path, before_is_s2, dst_transform, TARGET_CRS, dst_shape)
        slope = load_slope(slope_path, dst_transform, TARGET_CRS, dst_shape)
        slope_mask = slope >= MIN_SLOPE_DEG

        Xo = np.moveaxis(b4, 0, -1).reshape(-1, 4)
        Yo = np.moveaxis(a4, 0, -1).reshape(-1, 4)
        pm = valid.reshape(-1)
        conf_opt = np.zeros(h * w, dtype=np.float32)
        if np.sum(pm) > 100:
            _, p = irmad(Xo[pm], Yo[pm])
            conf_opt[pm] = p

        spre, spost = load_sar(sar_pre, sar_post, dst_transform, TARGET_CRS, dst_shape)
        Xs = np.moveaxis(spre, 0, -1).reshape(-1, 2)
        Ys = np.moveaxis(spost, 0, -1).reshape(-1, 2)
        conf_sar = np.zeros(h * w, dtype=np.float32)
        if np.sum(pm) > 100:
            _, p2 = irmad(Xs[pm], Ys[pm])
            conf_sar[pm] = p2

        conf_opt2 = conf_opt.reshape(h, w)
        conf_sar2 = conf_sar.reshape(h, w)
        fused = np.sqrt(np.clip(conf_opt2, 0, 1) * np.clip(conf_sar2, 0, 1)).astype(np.float32)
        fused *= slope_mask.astype(np.float32)

        feat = np.concatenate([fused[..., None], np.stack([norm01(a4[i]) for i in range(4)], axis=-1), norm01(slope)[..., None]], axis=-1)

        m_per_deg_lon = M_PER_DEG_LAT * max(np.cos(mean_lat_rad), 1e-6)
        px_area_m2 = abs(dst_transform.a * dst_transform.e) * (m_per_deg_lon * M_PER_DEG_LAT)
        px_per_segment = max(1, int((TARGET_SUPERPIXEL_M ** 2) / max(px_area_m2, 1e-6)))
        nseg = max(50, int((h * w) / px_per_segment))
        seg = slic(feat, n_segments=nseg, compactness=0.2, start_label=1, channel_axis=-1)

        seg_conf = np.zeros_like(fused)
        for sid in np.unique(seg):
            m = seg == sid
            seg_conf[m] = float(np.mean(fused[m]))

        # Proper hysteresis: keep connected 'low' regions only where they contain at least
        # one 'high' confidence seed pixel (skimage's tested/robust implementation).
        mask = apply_hysteresis_threshold(seg_conf, LOW_CONF, HIGH_CONF)
        mask = mask & slope_mask
        mask = remove_small_objects(mask, min_size=5)

        lbl = label(mask)
        px_area = px_area_m2
        cand = []
        for rg in regionprops(lbl, intensity_image=fused):
            area = rg.area * px_area
            if area < MIN_BLOB_AREA_M2 or area > MAX_BLOB_AREA_M2:
                continue
            maj = rg.major_axis_length or 1
            minr = rg.minor_axis_length or 1
            elong = maj / max(minr, 1e-6)
            if elong > MAX_ELONGATION:
                continue
            sev = float(rg.mean_intensity)
            score = float(area * sev)
            r0, c0, r1, c1 = rg.bbox
            lon_min, lat_max = dst_transform * (c0, r0)
            lon_max, lat_min = dst_transform * (c1, r1)
            cand.append({'incident_id': inc_id, 'area_m2': float(area), 'elongation': float(elong), 'severity': sev, 'score': score, 'bbox_lonlat': [float(lon_min), float(lat_min), float(lon_max), float(lat_max)], 'bbox_px': [int(r0), int(c0), int(r1), int(c1)]})

        cand = sorted(cand, key=lambda x: x['score'], reverse=True)[:MAX_CANDIDATES_PER_INCIDENT]

        pixel_size_m = max((px_area_m2 ** 0.5), 1e-6)
        pad_px = max(1, int(round(CHIP_PAD_M / pixel_size_m)))
        inc_dir_local = f'/kaggle/working/incident_{inc_id}'
        for cid, c in enumerate(cand, start=1):
            c['candidate_id'] = cid
            r0, c0, r1, c1 = c.pop('bbox_px')
            pr0, pc0 = max(0, r0 - pad_px), max(0, c0 - pad_px)
            pr1, pc1 = min(h, r1 + pad_px), min(w, c1 + pad_px)
            win = Window(col_off=pc0, row_off=pr0, width=pc1 - pc0, height=pr1 - pr0)
            chip_transform = window_transform(win, dst_transform)

            cand_dir_local = f'{inc_dir_local}/candidate_{cid}'
            os.makedirs(cand_dir_local, exist_ok=True)
            cand_repo_dir = f'{HF_CAND_ROOT}/incident_{inc_id}/candidate_{cid}'
            chip_specs = {
                'before': b4[:, pr0:pr1, pc0:pc1],
                'after': a4[:, pr0:pr1, pc0:pc1],
                'slope': slope[pr0:pr1, pc0:pc1],
                'mask': mask[pr0:pr1, pc0:pc1].astype(np.float32),
            }
            for chip_type, chip_arr in chip_specs.items():
                fn_name = f'incident_{inc_id}_candidate_{cid}_{chip_type}.tif'
                local_path = os.path.join(cand_dir_local, fn_name)
                write_chip(local_path, chip_arr, chip_transform, TARGET_CRS)
                pending_ops.append(CommitOperationAdd(path_in_repo=f'{cand_repo_dir}/{fn_name}', path_or_fileobj=local_path))

        if cand:
            pending_meta.append((inc_id, inc_dir_local))
            candidate_metadata.extend(cand)
        if len(pending_ops) >= UPLOAD_BATCH_SIZE:
            flush_pending()

        records.append({'incident_id': inc_id, 'status': 'ok', 'n_candidates': len(cand), 'error': ''})
        print(f'incident_{inc_id}: {len(cand)} candidates')
    except Exception as e:
        shutil.rmtree(f'/kaggle/working/incident_{inc_id}', ignore_errors=True)
        records.append({'incident_id': inc_id, 'status': 'failed', 'n_candidates': 0, 'error': str(e)})
        print(f'incident_{inc_id} failed: {e}')

flush_pending()

status_csv = '/kaggle/working/candidate_status.csv'
pd.DataFrame(records).to_csv(status_csv, index=False)
api.upload_file(path_or_fileobj=status_csv, path_in_repo=f'{HF_CAND_ROOT}/candidate_status.csv', repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION)

if candidate_metadata:
    meta_rows = [{
        'incident_id': c['incident_id'],
        'candidate_id': c['candidate_id'],
        'area_m2': c['area_m2'],
        'elongation': c['elongation'],
        'severity': c['severity'],
        'score': c['score'],
        'lon_min': c['bbox_lonlat'][0],
        'lat_min': c['bbox_lonlat'][1],
        'lon_max': c['bbox_lonlat'][2],
        'lat_max': c['bbox_lonlat'][3],
    } for c in candidate_metadata]
    meta_csv = '/kaggle/working/candidate_metadata.csv'
    pd.DataFrame(meta_rows).to_csv(meta_csv, index=False)
    api.upload_file(path_or_fileobj=meta_csv, path_in_repo=f'{HF_CAND_ROOT}/candidate_metadata.csv', repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION)

print('Candidate detection pass complete')
